# 13 — Optional: Eigenvalues, SVD, and Low-Rank Structure

**Master syllabus coverage:** Added mathematical foundation for V2 compression and LoRA

## Why this matters

Spectral structure explains dominant directions, matrix conditioning, compression, and low-rank adaptation. SVD provides a concrete bridge from linear algebra to modern model efficiency.

## Learning objectives

- Verify an eigendecomposition for a symmetric matrix.
- Interpret and reconstruct a rectangular matrix with SVD.
- Measure rank-k approximation error from discarded singular values.
- Relate factorized low-rank updates to LoRA parameter savings.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Eigenvectors reveal invariant directions

For square $A$, an eigenvector $v\ne0$ satisfies $Av=\lambda v$. The transformation
changes only its scale (and possibly sign/phase), not its direction. Not every matrix
has a full real orthogonal eigenbasis; symmetric real matrices do.


In [ ]:
import numpy as np
import torch

A = np.array([[3.0, 1.0], [1.0, 3.0]])
eigenvalues, eigenvectors = np.linalg.eigh(A)  # for symmetric matrices
print("eigenvalues:", eigenvalues)
for value, vector in zip(eigenvalues, eigenvectors.T):
    np.testing.assert_allclose(A @ vector, value * vector)
np.testing.assert_allclose(eigenvectors.T @ eigenvectors, np.eye(2), atol=1e-12)


## 2. SVD works for every rectangular matrix

$$A=U\Sigma V^\top$$

$V^\top$ rotates input coordinates, diagonal $\Sigma$ scales independent directions,
and $U$ rotates into output coordinates. Singular values are nonnegative and sorted.
The number of nonzero singular values is the matrix rank.


In [ ]:
rng = np.random.default_rng(42)
matrix = rng.normal(size=(6, 4))
U, singular_values, Vt = np.linalg.svd(matrix, full_matrices=False)
reconstructed = U @ np.diag(singular_values) @ Vt
np.testing.assert_allclose(reconstructed, matrix)
print("shapes:", U.shape, singular_values.shape, Vt.shape)
print("singular values:", singular_values.round(3))


## 3. Best low-rank approximation

Keeping the largest $k$ singular values produces the best rank-$k$ approximation under
Frobenius and spectral norms. The discarded singular values exactly characterize error:

$$\|A-A_k\|_F^2=\sum_{i>k}\sigma_i^2$$

This explains compression and why learned low-rank adapters can express useful updates
with far fewer trainable parameters.


In [ ]:
for k in range(1, len(singular_values) + 1):
    approx = U[:, :k] @ np.diag(singular_values[:k]) @ Vt[:k]
    measured = np.linalg.norm(matrix - approx, ord="fro")
    predicted = np.sqrt(np.sum(singular_values[k:] ** 2))
    np.testing.assert_allclose(measured, predicted, atol=1e-12)
    params = k * (matrix.shape[0] + matrix.shape[1])
    print(f"rank={k}: error={measured:.3f}, factor values={params}")


## 4. LoRA connection

A frozen weight $W\in\mathbb{R}^{d_{out}\times d_{in}}$ can receive a rank-$r$ update
$\Delta W=BA$, where $B=[d_{out},r]$ and $A=[r,d_{in}]$. Instead of training
$d_{out}d_{in}$ values, train $r(d_{out}+d_{in})$ values. Low rank is a constraint on
the update, not necessarily on the base weight.


In [ ]:
torch.manual_seed(42)
d_in, d_out, rank = 16, 12, 2
W = torch.randn(d_out, d_in)
A_factor = torch.randn(rank, d_in)
B_factor = torch.randn(d_out, rank)
delta = B_factor @ A_factor
x = torch.randn(5, d_in)

direct = x @ (W + delta).T
factored = x @ W.T + (x @ A_factor.T) @ B_factor.T
torch.testing.assert_close(direct, factored)
print("full update values:", W.numel(), "low-rank values:", A_factor.numel() + B_factor.numel())


## Failure modes to recognize

- **General eigensolver assumptions:** nonsymmetric matrices can have complex or defective decompositions.
- **Rank chosen only by parameter count:** approximation error or downstream quality becomes unacceptable.
- **Factor overhead ignored:** low-rank factors can exceed the original size when rank is not small.
- **Base/update confusion:** LoRA constrains the adaptation, not the pretrained matrix itself.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Create a known rank-2 matrix from two factors and confirm only two singular values are nonzero.
2. Plot reconstruction error versus rank for a noisy low-rank matrix.
3. Calculate the LoRA parameter ratio for a 4096×4096 weight at ranks 4, 8, and 64.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can reconstruct with SVD, choose a rank using an error tradeoff, and derive the cost of a low-rank update.

**Next:** 14 — Derivatives and local sensitivity.
